In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import scipy.stats as stats
from scipy.stats import shapiro, levene
from statsmodels.stats.multicomp import MultiComparison
import pingouin as pg
from matplotlib.patches import Patch
import scikit_posthocs as sp
from scipy.stats import chi2_contingency, chisquare, fisher_exact
from scipy.stats import chi2
import platform
import warnings
import statsmodels.api as sm
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import math
import xml.etree.ElementTree as ET
import os

warnings.filterwarnings('ignore')

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 관광인프라

In [19]:
관광인프라 = pd.read_csv('관광인프라.csv')

In [20]:
인프라점수 = pd.DataFrame(1 + 관광인프라.groupby('코스명')['거리가중치'].sum()).reset_index()
인프라점수['관광인프라'] = round(np.log(인프라점수['거리가중치']), 1)
인프라점수['산이름'] = 인프라점수['코스명'].str.split('_').str[0]
인프라점수 = 인프라점수[['코스명', '산이름', '관광인프라']]

# 주차장

In [21]:
사용가능_주차장 = pd.read_csv('사용가능_주차장.csv')
사용가능_주차장 = 사용가능_주차장.rename(columns={
    'mountain_name' : '산이름',
    'trail_code' : '코스명',
    'base_type' : '기준점',
    'place_name' : '장소명',
    'category' : '카테고리',
    'lat' : '위도',
    'lng' : '경도',
    'distance_m' : '거리(m)',
    'address' : '주소',
})

In [22]:
최단거리_주차장 = (사용가능_주차장
            .sort_values('거리(m)')
            .drop_duplicates(subset='코스명', keep='first')
            .reset_index(drop=True))

최단거리_주차장 = 최단거리_주차장[['코스명', '산이름', '장소명', '위도', '경도', '거리(m)']].sort_values('코스명').reset_index(drop=True)

In [23]:
점수_주차장 = 인프라점수.merge(최단거리_주차장[['코스명', '거리(m)']],
                 how = 'left',
                 on = '코스명'
                )

점수_주차장['주차장_접근성'] = 점수_주차장['거리(m)'].map(lambda x: 10 if x <= 400 else( 
                                                                8 if x <= 800 else( 
                                                                5 if x <= 1200 else(
                                                                2 if x <= 2000 else(
                                                                0 )))))
점수_주차장 = 점수_주차장.drop('거리(m)', axis = 1)

# 정류장

In [24]:
버스정류장_최종 = pd.read_csv('버스정류장_최종.csv')

In [25]:
버스정류장_최종

,코스명,유형설명,출발_정류장명,출발_도시코드,출발_정류장_위도,출발_정류장_경도,출발_정류장_거리_m,도착_정류장명,도착_도시코드,도착_정류장_위도,도착_정류장_경도,도착_정류장_거리_m
0,가리산_01,편도_하산,가리산휴양림,32310,37.861625,127.987215,471.0,가리산휴양림,32310,37.861625,127.987215,2607.5
1,가리산_02,등하산_횡단,가리산휴양림,32310,37.861625,127.987215,409.4,가리산휴양림,32310,37.861625,127.987215,1538.6
2,가리왕산_01,등하산_횡단,장구목이,32340,37.487273,128.583342,7.9,정선시외버스터미널,32350,37.379100,128.650533,8815.8
3,가리왕산_02,등하산_횡단,대화버스터미널,32340,37.500217,128.452800,8421.8,대화버스터미널,32340,37.500217,128.452800,7466.1
4,가야산_01,편도_등산,해인사,38400,35.792062,128.089125,499.3,법전2리,37380,35.850711,128.126084,3131.2
...,...,...,...,...,...,...,...,...,...,...,...,...
599,희양산_04,등하산_원점회귀,은티,33360,36.738104,127.989607,716.5,은티,33360,36.738104,127.989607,638.5
600,희양산_05,등하산_원점회귀,종산,33360,36.758899,127.977615,56.3,종산,33360,36.758899,127.977615,31.5
601,희양산_06,등하산_원점회귀,입석,33360,36.764764,127.957940,264.1,입석,33360,36.764764,127.957940,856.8
602,희양산_07,등하산_횡단,진촌,33360,36.748879,128.012345,256.0,진촌,33360,36.748879,128.012345,1032.5


In [26]:
점수_정류장 = 점수_주차장.merge(버스정류장_최종[['코스명', '출발_정류장_거리_m']],
                 how = 'left',
                 on = '코스명'
                )
점수_정류장['정류장_접근성'] = 점수_정류장['출발_정류장_거리_m'].map(lambda x: 10 if x <= 400 else( 
                                                    8 if x <= 800 else( 
                                                    5 if x <= 1200 else(
                                                    2 if x <= 2000 else(
                                                    0 )))))
점수_정류장 = 점수_정류장.drop('출발_정류장_거리_m', axis = 1)

In [27]:
점수_정류장

,코스명,산이름,관광인프라,주차장_접근성,정류장_접근성
0,가리산_01,가리산,3.7,10,8
1,가리산_02,가리산,3.5,10,8
2,가리왕산_01,가리왕산,3.8,0,10
3,가리왕산_02,가리왕산,2.3,0,0
4,가야산_01,가야산,5.3,10,8
...,...,...,...,...,...
601,희양산_04,희양산,3.9,5,8
602,희양산_05,희양산,4.4,2,10
603,희양산_06,희양산,4.4,0,10
604,희양산_07,희양산,4.2,2,10


# 매력

In [28]:
매력점수 = pd.read_csv('최종_매력지수.csv')

In [ ]:
# 산이름 통일
name_map = {
    '남산(금오산)': '남산',
    '내장산(신선봉)': '내장산',
    '덕숭산(수덕산)': '덕숭산',
    '덕유산(향적봉)': '덕유산',
    '도봉산(자운봉)': '도봉산',
    '마이산(암마이산)': '마이산',
    '방태산(주억봉)': '방태산',
    '변산(의상봉)': '변산',
    '북한산(백운대)': '북한산',
    '비슬산(천왕봉)': '비슬산',
    '설악산(대청봉)': '설악산',
    '오대산(비로봉)': '오대산',
    '운악산(현등산)': '운악산',
    '응봉산(매봉산)': '응봉산',
    '지리산(천왕봉)': '지리산(통영)'
}
score_cols = [
    '전망',
    '힐링',
    '사진',
    '등산로',
    '성취감',
    '계절매력'
]

# 컬럼명 한글화
매력점수 = 매력점수.rename(columns={
    'mountain_name' : '산이름',
    'total_score' : '매력종합점수',
    'view_score_weighted' : '전망',
    'healing_score_weighted' : '힐링',
    'sns_photo_score_weighted' : '사진',
    'trail_condition_score_weighted' : '등산로',
    'fun_achievement_score_weighted' : '성취감',
    'seasonal_attraction_score_weighted' : '계절매력'
})

매력점수['산이름'] = (
    매력점수['산이름']
    .replace(name_map)
)


매력점수['특출매력'] = (
    매력점수[score_cols].idxmax(axis=1)
)

row_idx = np.arange(len(매력점수))
col_idx = 매력점수.columns.get_indexer(매력점수['특출매력'])

매력점수['특출점수'] = 매력점수.values[row_idx, col_idx].astype(float)

In [30]:
매력점수

,산이름,매력종합점수,전망,힐링,사진,등산로,성취감,계절매력,특출매력,특출점수
0,가리산,7.3,7.8,7.3,7.5,6.9,7.1,7.2,전망,7.8
1,가리왕산,7.2,7.4,7.5,7.2,6.0,7.2,8.0,계절매력,8.0
2,가야산,7.7,8.4,7.4,7.7,6.9,7.8,8.0,전망,8.4
3,가지산,7.4,8.1,7.2,7.5,6.4,7.4,8.2,계절매력,8.2
4,감악산,7.1,7.7,7.0,7.7,6.4,6.9,7.0,전망,7.7
...,...,...,...,...,...,...,...,...,...,...
94,황매산,7.8,8.5,7.4,8.5,6.9,7.1,8.9,계절매력,8.9
95,황석산,7.0,7.4,7.0,7.3,5.7,7.3,7.2,전망,7.4
96,황악산,7.0,7.0,7.0,6.8,6.7,7.1,7.2,계절매력,7.2
97,황장산,6.9,7.8,6.8,7.0,5.4,7.2,7.1,전망,7.8


In [31]:
점수 = 점수_정류장.merge(매력점수,
                 how = 'left',
                 on = '산이름'
                )

점수

,코스명,산이름,관광인프라,주차장_접근성,정류장_접근성,매력종합점수,전망,힐링,사진,등산로,성취감,계절매력,특출매력,특출점수
0,가리산_01,가리산,3.7,10,8,7.3,7.8,7.3,7.5,6.9,7.1,7.2,전망,7.8
1,가리산_02,가리산,3.5,10,8,7.3,7.8,7.3,7.5,6.9,7.1,7.2,전망,7.8
2,가리왕산_01,가리왕산,3.8,0,10,7.2,7.4,7.5,7.2,6.0,7.2,8.0,계절매력,8.0
3,가리왕산_02,가리왕산,2.3,0,0,7.2,7.4,7.5,7.2,6.0,7.2,8.0,계절매력,8.0
4,가야산_01,가야산,5.3,10,8,7.7,8.4,7.4,7.7,6.9,7.8,8.0,전망,8.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
601,희양산_04,희양산,3.9,5,8,7.2,7.8,7.1,7.4,6.1,7.5,7.6,전망,7.8
602,희양산_05,희양산,4.4,2,10,7.2,7.8,7.1,7.4,6.1,7.5,7.6,전망,7.8
603,희양산_06,희양산,4.4,0,10,7.2,7.8,7.1,7.4,6.1,7.5,7.6,전망,7.8
604,희양산_07,희양산,4.2,2,10,7.2,7.8,7.1,7.4,6.1,7.5,7.6,전망,7.8


In [32]:
최종 = pd.read_csv('100대명산_최종.csv')
최종

,코스명,산이름,유형설명,최고고도_m,누적상승_m,편도거리_km,총거리_km,예상시간_분,예상시간,출발_lat,출발_lon,도착_lat,도착_lon,난이도,세부난이도,난이도점수
0,가리산_01,가리산,편도_하산,929.7,651.8,3.27,6.55,196,3시간 16분,37.864537,127.983319,37.879128,127.967446,초급,초급2,158.32
1,가리산_02,가리산,등하산_횡단,1037.4,760.0,6.47,6.47,205,3시간 25분,37.864428,127.984191,37.868292,127.971856,초급,초급3,169.91
2,가리왕산_01,가리왕산,등하산_횡단,1596.6,1160.6,9.79,9.79,321,5시간 21분,37.487309,128.583265,37.426318,128.570360,상급,상급2,258.28
3,가리왕산_02,가리왕산,등하산_횡단,1590.0,951.1,6.68,6.68,202,3시간 22분,37.466601,128.538330,37.463691,128.523798,중급,중급3,241.42
4,가야산_01,가야산,편도_등산,1416.9,1483.0,4.99,9.99,372,6시간 12분,35.793714,128.094273,35.822969,128.120127,최상급,최상급2,368.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599,희양산_04,희양산,등하산_원점회귀,821.0,877.9,6.60,6.60,229,3시간 49분,36.731875,127.987550,36.734587,127.983943,중급,중급3,230.55
600,희양산_05,희양산,등하산_원점회귀,787.5,682.5,5.56,5.56,186,3시간 06분,36.758418,127.977417,36.758713,127.977348,초급,초급2,149.26
601,희양산_06,희양산,등하산_원점회귀,863.4,868.2,8.58,8.58,260,4시간 20분,36.762413,127.958360,36.765355,127.948350,중급,중급2,209.13
602,희양산_07,희양산,등하산_횡단,918.3,898.1,5.51,5.51,210,3시간 30분,36.746672,128.011527,36.740453,128.017215,중급,중급2,213.07


In [33]:
종합 = 최종.merge(점수,
                how = 'left',
                on = ['코스명', '산이름']
                )
종합.columns

Index(['코스명', '산이름', '유형설명', '최고고도_m', '누적상승_m', '편도거리_km', '총거리_km', '예상시간_분',
       '예상시간', '출발_lat', '출발_lon', '도착_lat', '도착_lon', '난이도', '세부난이도', '난이도점수',
       '관광인프라', '주차장_접근성', '정류장_접근성', '매력종합점수', '전망', '힐링', '사진', '등산로', '성취감',
       '계절매력', '특출매력', '특출점수'],
      dtype='object')

In [34]:
점수.to_csv('점수.csv', index = False)
종합.to_csv('종합.csv', index = False)